In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
# !pip install -U "transformers>=4.39.0"
# !pip install peft bitsandbytes
# !pip install -U "trl>=0.8.3"
# !pip install datasets

In [ ]:
# !pip install tensorboard

In [ ]:
############################################## Matching dataset format

In [ ]:
#!pip install git+https://github.com/huggingface/transformers accelerate


  Cloning https://github.com/huggingface/transformers to c:\users\user\appdata\local\temp\pip-req-build-4c90791k
  Resolved https://github.com/huggingface/transformers to commit 2527f71a47b267f2ea7f4afc71a340106dd08809
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached huggingface_hub-0.30.2-py3-none-any.whl.metadata (13 kB)
  Using cached tokenizers-0.21.1-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
  Using cached safetensors-0.5.3-cp38-abi3-win_amd64.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
Using cached safetensors-0.5.3-cp38-abi3-win_amd64.whl (308 kB)
Using cached tokenizers-0.21.1-cp39-abi3-w

  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers 'C:\Users\User\AppData\Local\Temp\pip-req-build-4c90791k'


In [ ]:
#!pip install -q transformers qwen-vl-utils

In [ ]:
#!pip install trl
#!pip install peft
#!pip install -U bitsandbytes
#!pip install --upgrade bitsandbytes
#!pip install tqdm

In [ ]:
import os
import json
import torch
#from transformers import AutoTokenizer, AutoProcessor, TrainingArguments, LlavaForConditionalGeneration, BitsAndBytesConfig ###
from transformers import AutoTokenizer,AutoProcessor, Blip2Processor, Blip2ForConditionalGeneration, TrainingArguments, BitsAndBytesConfig #Qwen2VLProcessor,  ###
from trl import SFTTrainer, SFTConfig  # Import SFTConfig here
from peft import LoraConfig
from datasets import Dataset, DatasetDict
from PIL import Image
#from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy

from peft import prepare_model_for_kbit_training, get_peft_model

In [ ]:
import os
import json
from PIL import Image
from io import BytesIO
from datasets import Dataset

# Define paths
image_folder = r"C:\Users\User\Pictures\Thesis\VQA dataset added\combined-img"
train_json_path = r"C:\Users\User\Pictures\Thesis\VQA json files\training-split3\train.json" ###
val_json_path = r"C:\Users\User\Pictures\Thesis\VQA json files\training-split3\validation.json"
test_json_path = r"C:\Users\User\Pictures\Thesis\VQA json files\training-split3\test.json"

# train_json_path = r"C:\Users\User\Pictures\Thesis\VQA json files\training-split3\11cls bk\train.json" ###
# val_json_path = r"C:\Users\User\Pictures\Thesis\VQA json files\training-split3\11cls bk\validation.json"
# test_json_path = r"C:\Users\User\Pictures\Thesis\VQA json files\training-split3\11cls bk\test.json"

# train_json_path = r"C:\Users\User\Pictures\Thesis\VQA json files\training-split3\exp.json"
# val_json_path = r"C:\Users\User\Pictures\Thesis\VQA json files\training-split3\exp.json"
# test_json_path = r"C:\Users\User\Pictures\Thesis\VQA json files\training-split3\exp.json"

# Function to convert your dataset into the required format
def convert_vqa_to_dialog_format(json_path, image_folder):
    #with open(json_path, 'r') as f:
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    dialogues = []

    for item in data:
        image_name = item['image_name']
        image_path = os.path.join(image_folder, image_name)

        # Load image using PIL
        try:
            image = Image.open(image_path)
        except Exception as e:
            print(f"Error opening image {image_name}: {e}")
            continue  # Skip if the image can't be opened

        # Construct the dialogue exchanges
        dialogue = []

        # User asks the question and provides image (add image to user role)
        dialogue.append({
            "role": "user",
            "content": [
                {
                    "index": None,
                    "text": item['question'],
                    "type": "text"
                },
                {
                    "index": 0,  # This is added to match the reference format
                    "text": None,  # No text for image, just indicate it's an image
                    "type": "image"  # Indicate that it's an image content
                }
            ]
        })

        # Assistant answers the question
        dialogue.append({
            "role": "assistant",
            "content": [{
                "index": None,
                "text": item['answer'],
                "type": "text"
            }]
        })

        # Include the image as a PIL image object (this is what you need)
        dialogues.append({
            'messages': dialogue,
            'images': [image]  # Store image as PIL object
        })

    return dialogues

# Convert train and validation data
train_dialogues = convert_vqa_to_dialog_format(train_json_path, image_folder)
val_dialogues = convert_vqa_to_dialog_format(val_json_path, image_folder)
test_dialogues = convert_vqa_to_dialog_format(test_json_path, image_folder)

# Create datasets from the converted dialogues
train_dataset = Dataset.from_dict({
    'messages': [dialogue['messages'] for dialogue in train_dialogues],
    'images': [dialogue['images'] for dialogue in train_dialogues]
})

val_dataset = Dataset.from_dict({
    'messages': [dialogue['messages'] for dialogue in val_dialogues],
    'images': [dialogue['images'] for dialogue in val_dialogues]
})

test_dataset = Dataset.from_dict({
    'messages': [dialogue['messages'] for dialogue in test_dialogues],
    'images': [dialogue['images'] for dialogue in test_dialogues]
})

# Display the first item of the train dataset to verify
print(train_dataset[0]['messages'][0])
print(train_dataset[0]['messages'][1])

{'content': [{'index': None, 'text': 'What is the name for this disease?', 'type': 'text'}, {'index': 0, 'text': None, 'type': 'image'}], 'role': 'user'}
{'content': [{'index': None, 'text': 'melanoma', 'type': 'text'}], 'role': 'assistant'}


In [ ]:
print(test_dataset[299]['messages'][0]['content'][0]['text'])
print(test_dataset[299]['messages'][1]['content'][0]['text'])

What diagnostic tests are recommended?
Clinical diagnosis and varicella-zoster blood test.


In [ ]:
print(val_dataset[299]['messages'][0]['content'][0]['text'])
print(val_dataset[299]['messages'][1]['content'][0]['text'])

What diagnostic tests are recommended?
Clinical evaluation and blood tests for varicella-zoster virus.


In [ ]:
print(train_dataset)

Dataset({
    features: ['messages', 'images'],
    num_rows: 5838
})


In [ ]:
len(val_dataset)

714

In [ ]:
print(train_dataset[0]['messages'][0])
print(train_dataset[0]['messages'][1])
print(train_dataset[0]['images'])

{'content': [{'index': None, 'text': 'What is the name for this disease?', 'type': 'text'}, {'index': 0, 'text': None, 'type': 'image'}], 'role': 'user'}
{'content': [{'index': None, 'text': 'melanoma', 'type': 'text'}], 'role': 'assistant'}
[{'bytes': None, 'path': 'C:\\Users\\User\\Pictures\\Thesis\\VQA dataset added\\combined-img\\ISIC_7623407.jpg'}]


In [ ]:
print(val_dataset[0]['messages'][0])
print(val_dataset[0]['messages'][1])
print(val_dataset[0]['images'])

{'content': [{'index': None, 'text': 'What is the condition seen in the image?', 'type': 'text'}, {'index': 0, 'text': None, 'type': 'image'}], 'role': 'user'}
{'content': [{'index': None, 'text': 'Melanoma', 'type': 'text'}], 'role': 'assistant'}
[{'bytes': None, 'path': 'C:\\Users\\User\\Pictures\\Thesis\\VQA dataset added\\combined-img\\ISIC_7607829.jpg'}]


In [ ]:
print(train_dataset[0]['images'][0]['path'])
img_path= train_dataset[0]['images'][0]['path']
img = Image.open(img_path)
print(img)
#img.show()

C:\Users\User\Pictures\Thesis\VQA dataset added\combined-img\ISIC_7623407.jpg
<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=512x512 at 0x2877196E190>


In [ ]:
#model_id = "Qwen/Qwen2-VL-7B-Instruct"
#model_id = "llava-hf/llava-v1.6-mistral-7b-hf"
#model_id = "llava-hf/llava-1.5-7b-hf"
model_id = "Salesforce/blip2-opt-2.7b"

# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
# )
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
)


model = Blip2ForConditionalGeneration.from_pretrained(model_id,
                                                      quantization_config=quantization_config,
                                                      torch_dtype=torch.float16)
# model = LlavaForConditionalGeneration.from_pretrained(model_id,
#                                                       quantization_config=quantization_config,
#                                                       torch_dtype=torch.float16)
model

self.pre_quantized False


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Blip2ForConditionalGeneration(
  (vision_model): Blip2VisionModel(
    (embeddings): Blip2VisionEmbeddings(
      (patch_embedding): Conv2d(3, 1408, kernel_size=(14, 14), stride=(14, 14))
    )
    (encoder): Blip2Encoder(
      (layers): ModuleList(
        (0-38): 39 x Blip2EncoderLayer(
          (self_attn): Blip2Attention(
            (dropout): Dropout(p=0.0, inplace=False)
            (qkv): Linear4bit(in_features=1408, out_features=4224, bias=True)
            (projection): Linear4bit(in_features=1408, out_features=1408, bias=True)
          )
          (layer_norm1): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
          (mlp): Blip2MLP(
            (activation_fn): GELUActivation()
            (fc1): Linear4bit(in_features=1408, out_features=6144, bias=True)
            (fc2): Linear4bit(in_features=6144, out_features=1408, bias=True)
          )
          (layer_norm2): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
        )
      )
    )
    (post_layerno

In [ ]:

#LLAVA_CHAT_TEMPLATE = """A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. {% for message in messages %}{% if message['role'] == 'user' %}USER: {% else %}ASSISTANT: {% endif %}{% for item in message['content'] %}{% if item['type'] == 'text' %}{{ item['text'] }}{% elif item['type'] == 'image' %}<image>{% endif %}{% endfor %}{% if message['role'] == 'user' %} {% else %}{{eos_token}}{% endif %}{% endfor %}""" #####
#LLAVA_CHAT_TEMPLATE = """{% for message in messages %}{% if message['role'] == 'user' %}USER: {% else %}ASSISTANT: {% endif %}{% for item in message['content'] %}{% if item['type'] == 'text' %}{{ item['text'] }}{% elif item['type'] == 'image' %}<image>{% endif %}{% endfor %}{% if message['role'] == 'user' %} {% else %}{{eos_token}}{% endif %}{% endfor %}""" ###

#tokenizer = AutoTokenizer.from_pretrained(model_id)
#tokenizer.chat_template = LLAVA_CHAT_TEMPLATE
#processor = AutoProcessor.from_pretrained(model_id)
processor = Blip2Processor.from_pretrained(model_id)
#processor.tokenizer.padding_side = "right"
#processor.tokenizer = tokenizer

#print(processor.tokenizer.model_max_length)

class LLavaDataCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, examples):
        texts = []
        images = []
        for example in examples:
            #print(example)
            messages = example["messages"]
            # text = self.processor.tokenizer.apply_chat_template(
            #     messages, tokenize=False, add_generation_prompt=False
            # )
            # text = self.processor.apply_chat_template(
            #     messages, tokenize=False, add_generation_prompt=False
            # )
            text = f"Question: {messages[0]['content'][0]['text']} Answer:{messages[1]['content'][0]['text']} END"
            #text=f"<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>{example['messages'][0]['content'][0]['text']}<|im_end|>\n<|im_start|>assistant\n{example['messages'][1]['content'][0]['text']}"

            #print("text:",text)
            texts.append(text)
            #images.append(example["images"][0])
            images.append(Image.open(example["images"][0]['path']))

        #print("texts:",texts)
        #print("images:",images)

        batch = self.processor(images, texts, return_tensors="pt", padding=True)

        #batch = self.processor(images=images, text=texts, padding=True, truncation=True, max_length=32, return_tensors="pt") #256##
        #print(batch)
        #batch_text = self.processor.tokenizer(texts, return_tensors="pt", padding="max_length", truncation=True, max_length=256)###
        #batch_image = self.processor(images=images, return_tensors="pt")
        #batch = {**batch_text, **batch_image}


        labels = batch["input_ids"].clone()
        if self.processor.tokenizer.pad_token_id is not None:
            labels[labels == self.processor.tokenizer.pad_token_id] = -100
        batch["labels"] = labels

        return batch

data_collator = LLavaDataCollator(processor)



# training_args = TrainingArguments(
#     output_dir="./output-Alt",
#     #report_to="tensorboard",
#     learning_rate=1e-5,  #1.4e-5,
#     #lr_scheduler_type="constant",
#     per_device_train_batch_size=1, #4
#     gradient_accumulation_steps=1, #1
#     #gradient_clip_val= 1.0,  ####
#     warmup_steps=10,  ###
#     logging_steps=20,
#     num_train_epochs=1, #2
#     #push_to_hub=True,
#     #weight_decay=0.01,
#     #evaluation_strategy="no",  # Disable evaluation
#     #eval_steps=50,
#     gradient_checkpointing=True,
#     remove_unused_columns=False,
#     fp16=True, #True
#     bf16=False,  #False
#     save_steps=100,  # Save checkpoint every 200 steps (adjust based on your training duration)
#     save_total_limit=5,  # Keep only the last 3 checkpoints
# )

# MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"
# EPOCHS = 1
# BATCH_SIZE = 1
# GRADIENT_CHECKPOINTING = True,  # Tradeoff between memory efficiency and computation time.
# USE_REENTRANT = False,
# OPTIM = "paged_adamw_32bit"
# LEARNING_RATE = 2e-5
# LOGGING_STEPS = 50
# EVAL_STEPS = 50
# SAVE_STEPS = 50
# EVAL_STRATEGY = "steps"
# SAVE_STRATEGY = "steps"
# METRIC_FOR_BEST_MODEL="eval_loss"
# LOAD_BEST_MODEL_AT_END=True
# MAX_GRAD_NORM = 1
# WARMUP_STEPS = 0
# DATASET_KWARGS={"skip_prepare_dataset": True} # We have to put for VLMs
# REMOVE_UNUSED_COLUMNS = False # VLM thing
# MAX_SEQ_LEN=128
# NUM_STEPS = (283 // BATCH_SIZE) * EPOCHS

training_args = SFTConfig(
    output_dir="./output-Alt",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_checkpointing=True,
    learning_rate=2e-4, # 2e-4
    #lr_scheduler_type="constant",
    logging_steps=20,
    #eval_steps=20,
    eval_strategy="no", # "no"
    #eval_strategy="steps", # "no"
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    #metric_for_best_model="eval_loss",
    #load_best_model_at_end=True,
    max_grad_norm=1,
    #warmup_steps=60,
    dataset_kwargs={"skip_prepare_dataset": True},
    max_seq_length=500,
    remove_unused_columns = False,
    optim="paged_adamw_32bit",
)

# lora_config = LoraConfig(
#     r=64,
#     lora_alpha=16,
#     target_modules=["q_proj", "v_proj"]
# )
def find_all_linear_names(model):  ###
    cls = torch.nn.Linear
    lora_module_names = set()
    multimodal_keywords = ['multi_modal_projector', 'vision_model']
    for name, module in model.named_modules():
        if any(mm_keyword in name for mm_keyword in multimodal_keywords):
            continue
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])

    if 'lm_head' in lora_module_names: # needed for 16-bit
        lora_module_names.remove('lm_head')
    return list(lora_module_names)
print(find_all_linear_names(model))

lora_config = LoraConfig( ###
    r=64,  #64
    lora_alpha=64,  #64
    #lora_dropout=0.1,
    #target_modules=find_all_linear_names(model),
    #target_modules=['v_proj','q_proj'],  #['q_proj', 'v_proj', 'o_proj', 'up_proj', 'gate_proj', 'down_proj', 'k_proj'] ##
    #target_modules=['model.visual.blocks.*.attn.proj','model.visual.blocks.*.attn.qkv','q_proj', 'v_proj', 'o_proj', 'up_proj', 'gate_proj', 'down_proj', 'k_proj'],
    #target_modules=['mlp.0','mlp.2','qkv','proj','q_proj', 'v_proj', 'o_proj', 'up_proj', 'gate_proj', 'down_proj', 'k_proj'],  # ['qkv','proj',]
    target_modules = [
        'q_proj', 'v_proj', 'out_proj','k_proj',
        #'up_proj', 'gate_proj', 'down_proj',
        'query','key','value',
        'qkv', 'projection',
        'fc1', 'fc2',
        'dense',
    ],
    #modules_to_save=['v_proj','q_proj'],
    init_lora_weights='gaussian',
    #task_type="CAUSAL_LM",
)
model= prepare_model_for_kbit_training(model) ###
model= get_peft_model(model, lora_config) ###

print(model.print_trainable_parameters())

#train_dataset = train_dataset.shuffle(seed=72)  # 42
#test_dataset = test_dataset.shuffle(seed=250)  # 100
val_dataset_alt = val_dataset
#val_dataset_alt = val_dataset_alt.shuffle(seed=92) # 42
val_dataset_alt = val_dataset_alt.select(range(115))

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


['query', 'key', 'q_proj', 'fc1', 'k_proj', 'language_projection', 'value', 'dense', 'out_proj', 'v_proj', 'fc2']
trainable params: 166,625,280 || all params: 3,911,387,136 || trainable%: 4.2600
None


In [ ]:
print(len(val_dataset_alt))
print(val_dataset_alt[0])
type(val_dataset_alt)

115
{'messages': [{'content': [{'index': None, 'text': 'What is the name of this condition?', 'type': 'text'}, {'index': 0, 'text': None, 'type': 'image'}], 'role': 'user'}, {'content': [{'index': None, 'text': 'Chickenpox', 'type': 'text'}], 'role': 'assistant'}], 'images': [{'bytes': None, 'path': 'C:\\Users\\User\\Pictures\\Thesis\\VQA dataset added\\combined-img\\30_VI-chickenpox (26).jpg'}]}


datasets.arrow_dataset.Dataset

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    #eval_dataset=val_dataset_alt,
    peft_config=lora_config,
    #dataset_text_field="text",  # need a dummy field
    #tokenizer=tokenizer,
    data_collator=data_collator,
    #dataset_kwargs={"skip_prepare_dataset": True},
    #processing_class=processor.tokenizer,
)

trainer.can_return_loss = True

trainer.train()
#trainer.train(resume_from_checkpoint=True)  ## 158,957,568  47,742,976  68,829,184   166,525,280

No label_names provided for model class `PeftModel`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
C:\Users\User\.conda\envs\cuda-p3.9\lib\site-packages\torch\utils\checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
20,2.631000
40,0.819800
60,0.673800
80,0.622700
100,0.539800
120,0.623600
140,0.576000
160,0.588600
180,0.715000
200,0.491200


C:\Users\User\.conda\envs\cuda-p3.9\lib\site-packages\torch\utils\checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
C:\Users\User\.conda\envs\cuda-p3.9\lib\site-packages\torch\utils\checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
C:\Users\User\.conda\envs\cuda-p3.9\lib\site-packages\torch\utils\checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
C:\Users\User\.conda\envs\cuda-p3.9\lib\site-packages\torch\utils\checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
C:\Users\User\.conda\envs\cuda-p3.9\lib\site-packages\torch\utils\checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
C:\Users\User\.conda\envs\cuda-p3.9\lib\site-packages\torch\utils\checkpoint.py:87: UserWarning

TrainOutput(global_step=5838, training_loss=0.37671237851136674, metrics={'train_runtime': 4767.0595, 'train_samples_per_second': 1.225, 'train_steps_per_second': 1.225, 'total_flos': 1.9916871093624766e+19, 'train_loss': 0.37671237851136674})

In [ ]:
############## Save model
trainer.save_model(training_args.output_dir)

In [ ]:
model

In [ ]:
# At the end of training, save the processor as well
#output_dir = r"C:\Users\User\Desktop\jupiter\Thesis\LLAVA finetuning\llava-1.5-7b-hf-ft-mix-vsft"
# output_dir = "./output-Alt"
# processor.save_pretrained(output_dir)  # Save processor along with the model

# # Save model and tokenizer as well
# model.save_pretrained(output_dir)
# #tokenizer.save_pretrained(output_dir)

In [ ]:
############################ Fixed inference

In [ ]:
import torch
from PIL import Image
from transformers import AutoTokenizer,AutoProcessor, Blip2Processor, Blip2ForConditionalGeneration, BitsAndBytesConfig

# === 1. Setup ===
#model_id = r"C:\Users\User\Desktop\jupiter\Thesis\LLAVA finetuning\llava-1.5-7b-hf-ft-mix-vsft"
#model_id = "./output-Alt"
#model_id = r"C:\Users\User\Desktop\jupiter\Thesis\Qwen 2 VL finetuning\output-Alt"
model_id = "Salesforce/blip2-opt-2.7b"
#model_id = "llava-hf/llava-1.5-7b-hf"  ### To load base model

# === 2. Load model, tokenizer, and processor ===
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
# quant_config = BitsAndBytesConfig(
#     load_in_8bit=True,
#     bnb_8bit_quant_type="int8",
#     bnb_8bit_compute_dtype=torch.float16,
# )

model = Blip2ForConditionalGeneration.from_pretrained(model_id,
                                                      quantization_config=quant_config,
                                                      torch_dtype=torch.float16,)
#tokenizer = AutoTokenizer.from_pretrained(model_id)
#processor = AutoProcessor.from_pretrained(model_id)
processor = Blip2Processor.from_pretrained(model_id)
#processor.tokenizer.padding_side = "right"

# === 3. Apply chat template to tokenizer ===
#LLAVA_CHAT_TEMPLATE = """A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. {% for message in messages %}{% if message['role'] == 'user' %}USER: {% else %}ASSISTANT: {% endif %}{% for item in message['content'] %}{% if item['type'] == 'text' %}{{ item['text'] }}{% elif item['type'] == 'image' %}<image>{% endif %}{% endfor %}{% if message['role'] == 'user' %} {% else %}{{eos_token}}{% endif %}{% endfor %}"""
#LLAVA_CHAT_TEMPLATE = """{% for message in messages %}{% if message['role'] == 'user' %}USER: {% else %}ASSISTANT: {% endif %}{% for item in message['content'] %}{% if item['type'] == 'text' %}{{ item['text'] }}{% elif item['type'] == 'image' %}<image>{% endif %}{% endfor %}{% if message['role'] == 'user' %} {% else %}{{eos_token}}{% endif %}{% endfor %}"""
#tokenizer.chat_template = LLAVA_CHAT_TEMPLATE
#processor.tokenizer = tokenizer


self.pre_quantized False


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [ ]:
# === 3. Apply chat template to tokenizer ===
#LLAVA_CHAT_TEMPLATE = """A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. {% for message in messages %}{% if message['role'] == 'user' %}USER: {% else %}ASSISTANT: {% endif %}{% for item in message['content'] %}{% if item['type'] == 'text' %}{{ item['text'] }}{% elif item['type'] == 'image' %}<image>{% endif %}{% endfor %}{% if message['role'] == 'user' %} {% else %}{{eos_token}}{% endif %}{% endfor %}"""
# LLAVA_CHAT_TEMPLATE = """{% for message in messages %}{% if message['role'] == 'user' %}USER: {% else %}ASSISTANT: {% endif %}{% for item in message['content'] %}{% if item['type'] == 'text' %}{{ item['text'] }}{% elif item['type'] == 'image' %}<image>{% endif %}{% endfor %}{% if message['role'] == 'user' %} {% else %}{{eos_token}}{% endif %}{% endfor %}"""
# tokenizer = AutoTokenizer.from_pretrained(model_id)
# processor = AutoProcessor.from_pretrained(model_id)
# tokenizer.chat_template = LLAVA_CHAT_TEMPLATE
# processor.tokenizer = tokenizer

In [ ]:
print(f"Before adapter parameters: {model.num_parameters()}")
model.load_adapter("./output-Alt")
print(f"After adapter parameters: {model.num_parameters()}")

Before adapter parameters: 3744761856
After adapter parameters: 3911387136


In [ ]:
# === 4. Load image ===
image_path = r"C:\Users\User\Pictures\Thesis\VQA dataset added\combined-img\ISIC_0025707.jpg"
#image_path = r"C:\Users\User\Pictures\Brac\Transformer-apps.jpg"
#image_path = r"C:\Users\User\Pictures\Screenshots\Screenshot (140).png"
#image_path = r"C:\Users\User\Pictures\Thesis\1) kaggle dataset\IMG_CLASSES\3. Atopic Dermatitis - 1.25k\1_60.jpg"###
#image_path = r"D:\images\CUX7OA.jpg"
#image_path = r"D:\images\mug.jpg"
#image_path = r"D:\images\Mozi.jpg"
##ISIC_0025452.jpg  V
##ISIC_0025707.jpg  V
## 8_VI-chickenpox (1).jpg
## 60_VI-chickenpox (12).jpg
##ISIC_7614036.jpg  M
##ISIC_7613461.jpg  M
image = Image.open(image_path)

# === 5. Prepare message in chat format ===
# messages = [
#     {
#         "role": "user",
#         "content": [
#             {"type": "text", "text": "What is the name of the disease?"}, #what causes this?
#             {"type": "image"}
#         ]
#     }
# ]

# === 6. Generate chat-formatted prompt ===
#print("prompt=", tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
#prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

#prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#prompt = processor.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#prompt= USER: what tests are needed?<image>
#prompt= "USER: Explain the image<image>"
prompt = "Question: What is the cause of this disease? Answer:"
#question = "What is the name of the disease?"
#prompt=f"<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>{question}<|im_end|>\n<|im_start|>assistant\n"

# === 7. Prepare inputs ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#model = model.to(device)

#print("inputs=",processor(image, prompt, return_tensors="pt").to(device))
inputs = processor(image, prompt, return_tensors="pt").to(device)

# === 8. Generate output ===
with torch.no_grad():
    #print("outputs:", model.generate(**inputs, max_new_tokens=500))
    outputs = model.generate(**inputs, max_new_tokens=50)

# === 9. Decode and print output ===
#response = tokenizer.decode(outputs[0], skip_special_tokens=True)
output_text = processor.batch_decode(outputs, skip_special_tokens=True)[0].strip()
del inputs

#print(output_text)
print("--------------------")
# if("assistant\n" in output_text):
#     output_text = output_text.split("assistant\n")[-1].strip()

#print("USER:", messages[0]["content"][0]["text"]) # print question

print("raw out", output_text)

output_text = output_text.split("Answer:")[1]
output_text = output_text.split("Question:")[0].strip()
output_text = output_text.split("END")[0].strip()
print("\n🧠 Model Response:", output_text) # print response
#print("\n🧠 Model Response:", response) # print response


--------------------
raw out Question: What is the cause of this disease? Answer:Vascular lesions are caused by abnormalities in the blood vessels END END END END END END END END END END END END END END END END END END END END END END END END END END END END END END END END END END END END END END END

🧠 Model Response: Vascular lesions are caused by abnormalities in the blood vessels


In [ ]:
#!pip install bert_score

In [ ]:
############################### Bert Score
import torch
from PIL import Image
from transformers import Blip2Processor, Blip2ForConditionalGeneration, BitsAndBytesConfig
from bert_score import score  # You need to install this: pip install bert-score
from tqdm import tqdm

# === 1. Load model and processor ===
#model_id = r"C:\Users\User\Desktop\jupiter\Thesis\LLAVA finetuning\llava-1.5-7b-hf-ft-mix-vsft"
model_id = "Salesforce/blip2-opt-2.7b"  ### To load base model
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model =  Blip2ForConditionalGeneration.from_pretrained(model_id,
                                                      quantization_config=quant_config,
                                                      torch_dtype=torch.float16)

print(f"Before adapter parameters: {model.num_parameters()}")
model.load_adapter("./output-Alt")
print(f"After adapter parameters: {model.num_parameters()}")

#tokenizer = AutoTokenizer.from_pretrained(model_id)
processor = Blip2Processor.from_pretrained(model_id)
processor.tokenizer.padding_side = "right"

# === 2. Set custom chat template ===
#LLAVA_CHAT_TEMPLATE = """{% for message in messages %}{% if message['role'] == 'user' %}USER: {% else %}ASSISTANT: {% endif %}{% for item in message['content'] %}{% if item['type'] == 'text' %}{{ item['text'] }}{% elif item['type'] == 'image' %}<image>{% endif %}{% endfor %}{% if message['role'] == 'user' %} {% else %}{{eos_token}}{% endif %}{% endfor %}"""
#tokenizer.chat_template = LLAVA_CHAT_TEMPLATE
#processor.tokenizer = tokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#model.to(device)

# === 3. Load your val_dataset ===
# import json
# with open("val_dataset.json", "r") as f:
#     val_dataset = json.load(f)


self.pre_quantized False


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Before adapter parameters: 3744761856


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


After adapter parameters: 3911387136


In [ ]:
# === 4. Generate predictions and compare ===
references = []
predictions = []

#print(type(val_dataset))
#print("val_dataset[0]=", val_dataset[0])
#print("val_dataset[0][messages]:", val_dataset[0]["messages"])
for item in tqdm(test_dataset):  # limit to first 50 samples for faster testing
    try:
        #print("item:", item)
        #print("item keys:", item.keys())
        #print(item["messages"])

        messages = item["messages"]
        image_path = item["images"][0]["path"]
        image = Image.open(image_path)

        #prompt = processor.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
        prompt = f"Question: {messages[0]['content'][0]['text']} Answer:"
        #print("messages:", messages)
        #print("prompt:", prompt)
        inputs = processor(image, prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=32)

        #generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        output_text = processor.batch_decode(outputs, skip_special_tokens=True)[0].strip()
        del inputs
        # if("assistant\n" in output_text):
        #     output_text = output_text.split("assistant\n")[-1].strip()
        print("raw_out", output_text)
        output_text = output_text.split("Answer:")[1]
        output_text = output_text.split("Question:")[0].strip()
        output_text = output_text.split("END")[0].strip()

        true_answer = messages[1]["content"][0]["text"].strip()



        print("output_text:", output_text)
        print("true_answer:", true_answer)
        predictions.append(output_text)
        references.append(true_answer)

    except Exception as e:
        print(f"Skipping sample due to error: {e}")
        continue

#print(predictions[:10], references[:10])
# === 5. Evaluate using BERTScore ===
P, R, F1 = score(predictions, references, lang="en", verbose=True)
print(f"\nAverage BERTScore F1: {F1.mean().item():.4f}")
## Blip2 score:
# base model bertScore=%
#Average BERTScore F1: 0.9569
#Average Precision: 0.9549
#Average Recall: 0.9593

#Average BERTScore F1: 0.9561   0.9326  0.9340(warmup 60)  0.9236(128,128, lr5)
#Average Precision: 0.9562
#Average Recall: 0.9565

  0%|          | 1/721 [00:01<21:55,  1.83s/it]

raw_out Question: What is the name for this disease? Answer:Melanoma END END END END END END END END END END END END END END END END END END END END END END END END END END END END END
output_text: Melanoma
true_answer: Melanoma


  0%|          | 2/721 [00:03<20:50,  1.74s/it]

raw_out Question: Is it cancerous or benign? Answer:Cancerous END END END END END END END END END END END END END END END END END END END END END END END END END END END END END
output_text: Cancerous
true_answer: Cancerous


  0%|          | 3/721 [00:05<20:29,  1.71s/it]

...

calculating scores...
computing bert embedding.


  0%|          | 0/8 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/12 [00:00<?, ?it/s]

done in 0.81 seconds, 885.53 sentences/sec

Average BERTScore F1: 0.9412


In [ ]:
print(predictions[:10], references[:10])

In [ ]:
P, R, F1 = score(predictions, references, model_type="roberta-large", lang="en", verbose=True)###
#P, R, F1 = score(predictions, references, model_type="bert-base-uncased", lang="en", verbose=True)###
#P, R, F1 = score(predictions, references, model_type="microsoft/deberta-xlarge-mnli", lang="en", verbose=True)###
#P, R, F1 = score(predictions, references, model_type="dmis-lab/biobert-base-cased-v1.1", num_layers=12, lang="en", verbose=True)###

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


100%|██████████| 4/4 [00:00<00:00, 12.54it/s]


computing greedy matching.


100%|██████████| 7/7 [00:00<00:00, 76.09it/s]

done in 0.42 seconds, 986.53 sentences/sec


In [ ]:
print(f"\nAverage BERTScore F1: {F1.mean().item():.4f}")
print(f"\nAverage Precision: {P.mean().item():.4f}")
print(f"\nAverage Recall: {R.mean().item():.4f}")


Average BERTScore F1: 0.9427

Average Precision: 0.9427

Average Recall: 0.9430


In [ ]:
#!pip install nltk

  Using cached nltk-3.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
Using cached nltk-3.9.1-py3-none-any.whl (1.5 MB)
Using cached click-8.1.8-py3-none-any.whl (98 kB)


In [ ]:
############################## Bleu Score
############################### BLEU Score Evaluation
import torch
from PIL import Image
from tqdm import tqdm
from transformers import Blip2Processor, Blip2ForConditionalGeneration, BitsAndBytesConfig
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
#nltk.download('punkt')  # In case you use nltk word_tokenize (optional)

# === 1. Load model and processor ===
#output_dir = r"C:\Users\User\Desktop\jupiter\Thesis\LLAVA finetuning\llava-1.5-7b-hf-ft-mix-vsft"
model_id = "Salesforce/blip2-opt-2.7b"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = Blip2ForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quant_config,
    torch_dtype=torch.float16
)
print(f"Before adapter parameters: {model.num_parameters()}")
model.load_adapter("./output-Alt")
print(f"After adapter parameters: {model.num_parameters()}")

#tokenizer = AutoTokenizer.from_pretrained(output_dir)
processor = Blip2Processor.from_pretrained(model_id)

self.pre_quantized False


Loading checkpoint shards: 100%|██████████| 2/2 [00:22<00:00, 11.23s/it]


Before adapter parameters: 3744761856


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


After adapter parameters: 3911387136


In [ ]:

# === 2. Chat template ===
#LLAVA_CHAT_TEMPLATE = """{% for message in messages %}{% if message['role'] == 'user' %}USER: {% else %}ASSISTANT: {% endif %}{% for item in message['content'] %}{% if item['type'] == 'text' %}{{ item['text'] }}{% elif item['type'] == 'image' %}<image>{% endif %}{% endfor %}{% if message['role'] == 'user' %} {% else %}{{eos_token}}{% endif %}{% endfor %}"""
#tokenizer.chat_template = LLAVA_CHAT_TEMPLATE
#processor.tokenizer = tokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# === 3. Evaluate BLEU ===
bleu_scores = []
smooth = SmoothingFunction().method4  # Helps avoid zero scores for short sentences

for item in tqdm(val_dataset):
    try:
        messages = item["messages"]
        image_path = item["images"][0]["path"]
        image = Image.open(image_path)

        #prompt = processor.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
        prompt = f"Question: {messages[0]['content'][0]['text']} Answer:"
        inputs = processor(image, prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=32)

        output_text = processor.batch_decode(outputs, skip_special_tokens=True)[0].strip()

        # if("assistant\n" in output_text):
        #     output_text = output_text.split("assistant\n")[-1].strip()
        output_text = output_text.split("Answer:")[1]
        output_text = output_text.split("Question:")[0].strip()
        output_text = output_text.split("END")[0].strip()

        true_answer = messages[1]["content"][0]["text"].strip()

        # Tokenize
        true_answer_tokens = true_answer.split()
        output_text_tokens = output_text.split()

        print("true ans:",true_answer_tokens)
        print("output:",output_text_tokens)

        # BLEU-1 to BLEU-4 score (cumulative)
        score_bleu = sentence_bleu([true_answer_tokens], output_text_tokens, smoothing_function=smooth)
        bleu_scores.append(score_bleu)

    except Exception as e:
        print(f"Skipping sample due to error: {e}")
        continue

# === 4. Report BLEU
average_bleu = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0
print(f"\nAverage BLEU Score: {average_bleu:.4f}")
# Blip2 score:
# Average BLEU Score: 0.4255

  0%|          | 1/410 [00:02<13:43,  2.01s/it]

true ans: ['Melanoma']
output: ['Melanoma']


  0%|          | 2/410 [00:03<13:09,  1.93s/it]

true ans: ['yes']
output: ['Cancerous.']


  1%|          | 3/410 [00:05<12:19,  1.82s/it]

...

true ans: ['Vaccination', 'and', 'avoiding', 'contact', 'with', 'infected', 'individuals.']
output: ['Vaccination', 'and', 'avoiding', 'contact', 'with', 'infected', 'individuals.']


100%|█████████▉| 409/410 [11:57<00:01,  1.77s/it]

true ans: ['Clinical', 'diagnosis', 'and', 'varicella-zoster', 'test.']
output: ['Clinical', 'evaluation', 'and', 'varicella-zoster', 'PCR', 'test.']


100%|██████████| 410/410 [11:59<00:00,  1.75s/it]

true ans: ['Viral']
output: ['Viral']

Average BLEU Score: 0.4255


In [ ]:
###################################### SBERT score

In [ ]:
#!pip install sentence-transformers

  Using cached joblib-1.4.2-py3-none-any.whl.metadata (5.4 kB)
   ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
    --------------------------------------- 0.3/11.2 MB ? eta -:--:--
   --- ------------------------------------ 1.0/11.2 MB 2.5 MB/s eta 0:00:05
   ----- ---------------------------------- 1.6/11.2 MB 2.8 MB/s eta 0:00:04
   --------- ------------------------------ 2.6/11.2 MB 3.4 MB/s eta 0:00:03
   ------------ --------------------------- 3.4/11.2 MB 3.5 MB/s eta 0:00:03
   --------------- ------------------------ 4.2/11.2 MB 3.5 MB/s eta 0:00:02
   ----------------- ---------------------- 5.0/11.2 MB 3.7 MB/s eta 0:00:02
   -------------------- ------------------- 5.8/11.2 MB 3.6 MB/s eta 0:00:02
   ----------------------- ---------------- 6.6/11.2 MB 3.7 MB/s eta 0:00:02
   -------------------------- ------------- 7.3/11.2 MB 3.7 MB/s eta 0:00:02
   ----------------------------- ---------- 8.1/11.2 MB 3.6 MB/s eta 0:00:01
   ------------------------

In [ ]:
import torch
from PIL import Image
from tqdm import tqdm
from transformers import Blip2Processor, Blip2ForConditionalGeneration, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, util

# === 1. Load model and processor ===
#output_dir = r"C:\Users\User\Desktop\jupiter\Thesis\LLAVA finetuning\llava-1.5-7b-hf-ft-mix-vsft"
model_id = "Salesforce/blip2-opt-2.7b"
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = Blip2ForConditionalGeneration.from_pretrained(model_id,
                                                      quantization_config=quant_config,
                                                      torch_dtype=torch.float16)

print(f"Before adapter parameters: {model.num_parameters()}")
model.load_adapter("./output-Alt")
print(f"After adapter parameters: {model.num_parameters()}")

#tokenizer = AutoTokenizer.from_pretrained(output_dir)
processor = Blip2Processor.from_pretrained(model_id)
processor.tokenizer.padding_side = "right"

# === 2. Set custom chat template ===
#LLAVA_CHAT_TEMPLATE = """{% for message in messages %}{% if message['role'] == 'user' %}USER: {% else %}ASSISTANT: {% endif %}{% for item in message['content'] %}{% if item['type'] == 'text' %}{{ item['text'] }}{% elif item['type'] == 'image' %}<image>{% endif %}{% endfor %}{% if message['role'] == 'user' %} {% else %}{{eos_token}}{% endif %}{% endfor %}"""
#tokenizer.chat_template = LLAVA_CHAT_TEMPLATE
#processor.tokenizer = tokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#model = model.to(device)

# === 3. Load SBERT model for semantic similarity ===
#sbert_model = SentenceTransformer('all-MiniLM-L6-v2').to(device)
#sbert_model = SentenceTransformer('all-mpnet-base-v2').to(device)
sbert_model = SentenceTransformer('all-mpnet-base-v2').to(device)

# === 4. Evaluation ===
scores = []
count = 0

for item in tqdm(val_dataset):
    try:
        messages = item["messages"]
        image_path = item["images"][0]["path"]
        image = Image.open(image_path)

        # Build prompt from messages
        #prompt = processor.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
        prompt = f"Question: {messages[0]['content'][0]['text']} Answer:"
        inputs = processor(image, prompt, return_tensors="pt").to(device)

        # Generate prediction
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=32)
        output_text = processor.batch_decode(outputs, skip_special_tokens=True)[0].strip()
        del inputs
        # if("assistant\n" in output_text):
        #     output_text = output_text.split("assistant\n")[-1].strip()

        output_text = output_text.split("Answer:")[1]
        output_text = output_text.split("Question:")[0].strip()
        output_text = output_text.split("END")[0].strip()

        # Ground truth
        true_answer = messages[1]["content"][0]["text"].strip()

        # Encode with SBERT
        emb1 = sbert_model.encode(output_text, convert_to_tensor=True)
        emb2 = sbert_model.encode(true_answer, convert_to_tensor=True)

        # Compute cosine similarity
        similarity = util.pytorch_cos_sim(emb1, emb2).item()
        scores.append(similarity)
        count += 1

    except Exception as e:
        print(f"Skipping sample due to error: {e}")
        continue

# === 5. Final Result ===
avg_score = sum(scores) / len(scores) if scores else 0
print(f"\n🔍 Average SBERT Cosine Similarity: {avg_score:.4f} ({avg_score * 100:.2f}% semantic similarity)")


self.pre_quantized False


Loading checkpoint shards: 100%|██████████| 2/2 [00:19<00:00,  9.56s/it]


Before adapter parameters: 3744761856


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


After adapter parameters: 3911387136


100%|██████████| 410/410 [11:52<00:00,  1.74s/it]


🔍 Average SBERT Cosine Similarity: 0.8521 (85.21% semantic similarity)


In [ ]:
# LLaVA 1.5
### Average SBERT Cosine Similarity: 0.8189 (81.89% semantic similarity) (all-MiniLM-L6-v2)
### Average SBERT Cosine Similarity: 0.8266 (82.66% semantic similarity)  all-mpnet-base-v2
### Average SBERT Cosine Similarity: 0.8254 (82.54% semantic similarity) (pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb)

# Qwen 2.5 3b
### Average SBERT Cosine Similarity: 0.8469 (84.69% semantic similarity)  (all-mpnet-base-v2)

# Blip2 2.7b
### Average SBERT Cosine Similarity: 0.8521 (85.21% semantic similarity)  (all-mpnet-base-v2)